# Does splitting a batch preserve the update?

Chapter 9 · Day 12

Test the token denominator and the restart boundary. The experiment fixes dropout, initialization and data so a mismatch reveals implementation behavior.

Use the cycle: prediction → inspect mechanism → change one choice → interpret limits. Runnable reference answers follow each question. All computations use CPU; no model download or server is started.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
sys.path.insert(0, str(root / "src"))
import math, copy, json
import numpy as np
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "font.family": "DejaVu Sans", "font.size": 11})
from dongxi_llms import evaluation_lab as ev
from dongxi_llms import instruction_data_lab as data
from dongxi_llms import sft_lab as sft
def show_flow(labels, focus):
    fig, ax = plt.subplots(figsize=(11, 1.9))
    for i, label in enumerate(labels):
        ax.text(i, .5, label, ha="center", va="center",
                bbox=dict(boxstyle="round,pad=.45", fc="#dce9f5" if i == focus else "#f6f6f6", ec="#666"))
        if i: ax.annotate("", (i-.25,.5), (i-.75,.5), arrowprops=dict(arrowstyle="->",color="#555"))
    ax.set_xlim(-.6,len(labels)-.4); ax.set_ylim(0,1); ax.axis("off")
    ax.set_title("Schematic process: highlighted component is this lesson's focus")
    plt.show()


In [ ]:
show_flow(["Microbatches","Token counts","Accumulated gradient","Optimizer state"], 2)

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![02 accumulation and checkpoint identity: reference plot 01](../figures/chapter-09/day-12-02_accumulation_and_checkpoint_identity-01.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

### Prediction

## Predict 1

If one microbatch has two answer targets and another six, should we average their mean losses equally to reproduce the joint token mean?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
examples=data.instruction_fixture()
batches=[data.collate(examples[:1]),data.collate(examples[1:])]
joint=sft.accumulated_gradients(sft.make_model(),[data.collate(examples)])
split=sft.accumulated_gradients(sft.make_model(),batches)
differences={name:float((joint[name]-split[name]).abs().max()) for name in joint}
assert max(differences.values())<2e-6
wrong=sft.make_model(); wrong.zero_grad()
for batch in batches:
    total,count=sft.token_loss_sum(wrong(batch["input_ids"]),batch["labels"]); (total/count/2).backward()
wrong_diff={name:float((joint[name]-p.grad).norm()) for name,p in wrong.named_parameters()}
fig,ax=plt.subplots(figsize=(9,4)); ax.barh(list(wrong_diff),list(wrong_diff.values()),color="#b16b31")
ax.set(xlabel="L2 gradient error against joint token mean",title="Deliberate failure: mean of unequal microbatch means")
plt.tight_layout(); plt.show()
assert max(wrong_diff.values())>1e-5

### Reference explanation

### Reference answer and mechanism

No. Accumulate each summed loss divided by the total eight targets. Equal means overweights the short microbatch.


**Saved reference preview — rerun the preceding plot cell for current inputs.**

![02 accumulation and checkpoint identity: reference plot 02](../figures/chapter-09/day-12-02_accumulation_and_checkpoint_identity-02.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

### Prediction

## Predict 2

Are model weights alone enough to reproduce the next AdamW update?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
batch=data.collate(examples)
model=sft.make_model(); optimizer=torch.optim.AdamW(model.parameters(),lr=.01)
def update(model,optimizer):
    optimizer.zero_grad(); total,count=sft.token_loss_sum(model(batch["input_ids"]),batch["labels"])
    (total/count).backward(); optimizer.step()
update(model,optimizer)
restored=sft.make_model(); restored.load_state_dict(copy.deepcopy(model.state_dict()))
restored_optimizer=torch.optim.AdamW(restored.parameters(),lr=.01)
restored_optimizer.load_state_dict(copy.deepcopy(optimizer.state_dict()))
update(model,optimizer); update(restored,restored_optimizer)
max_error=max(float((p-q).abs().max().detach()) for p,q in zip(model.parameters(),restored.parameters()))
assert max_error==0
print("Restarted next-update max absolute error:",max_error)

### Reference explanation

### Reference answer and mechanism

No. AdamW's moments and step counters affect the next update. Copy optimizer state as well as weights, then compare the same next batch.


### Prediction

## Predict 3

What does the equality above leave untested?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
metadata={"seed":1212,"backend":"CPU transparent decoder","data_hash":ev.canonical_hash([x.ids for x in examples]),
          "objective":"summed answer NLL / valid answer count","restart_boundary":"after optimizer step"}
print(json.dumps(metadata,indent=2))

### Reference explanation

### Reference answer and mechanism

It verifies one CPU local continuation with the same data/backend. It does not cover distributed shuffling, different hardware, scheduler state, tokenizer changes or a changed attention kernel.

## Reading the evidence

The figures are regenerated from the current lesson tensors or declared fixture. Axes and counts define the comparison. Change one control, rerun the relevant cell, and explain what changed in the mechanism. Separate a verified implementation property from a claim about a real language model.

Return to the corresponding chapter and worked solution for the complete argument.
